In [14]:
import os
import re
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from shapely.ops import unary_union
from pathlib import Path
import numpy as np

# ================= USER CONFIG =================
# 单文件调试：你只需要改这里

CS2_FILE = r"C:\Users\TJ002\Desktop\CS2_S1_result\filter\2023\gpkg\CS_OFFL_SIR_SIN_1B_20230424T135429_20230424T135644_E001_classified.gpkg"
S1_FILE  = r"C:\Users\TJ002\Desktop\CS2_S1_result\filter\2023\tif\S1A_EW_GRDM_1SDH_20230424T121731_20230424T121835_048238_05CCE4_C830_EW_HH_HV_int16x100_nodata-9999-0000000000-0000000000_processed_classified.tif"

# neighborhood window 半径（像素）
# 20 → 覆盖约 800–1000m 区域
WINDOW_RADIUS = 20

# S1 分类编码
S1_BACKGROUND = 0
S1_ICE = 1
S1_LEAD = 2
S1_REFR = 3

print("Config loaded.")

# Load CS2 GPKG
cs2_gdf = gpd.read_file(CS2_FILE)

print("CS2 loaded.")
print("CRS:", cs2_gdf.crs)
print(cs2_gdf.head())


Config loaded.
CS2 loaded.
CRS: EPSG:4326
    latitude  longitude                utc_time    pp_calc   std   kurtosis  \
0  76.131920 -69.698296 2023-04-24 13:55:05.629  42.029611  0.53  21.059999   
1  76.134627 -69.700161 2023-04-24 13:55:05.675  41.998258  1.22   7.520000   
2  76.137334 -69.702027 2023-04-24 13:55:05.720  46.293273  0.52  22.299999   
3  76.140041 -69.703894 2023-04-24 13:55:05.765  55.354159  0.71  15.750000   
4  76.142747 -69.705761 2023-04-24 13:55:05.811  49.136758  0.52  24.559999   

  class                    geometry  
0  lead  POINT (-69.69830 76.13192)  
1  lead  POINT (-69.70016 76.13463)  
2  lead  POINT (-69.70203 76.13733)  
3  lead  POINT (-69.70389 76.14004)  
4  lead  POINT (-69.70576 76.14275)  


In [15]:
# auto-detect classification column
class_col = next((c for c in ["class", "Class", "CLASS", "classification"] if c in cs2_gdf.columns), None)

if not class_col:
    raise ValueError("No classification column found in CS2 file.")

cs2_gdf[class_col] = cs2_gdf[class_col].str.lower().str.strip()

cs2_gdf[class_col].value_counts()


ice     2301
lead     125
Name: class, dtype: int64

In [16]:
count_lead      = (cs2_gdf[class_col] == "lead").sum()
count_ice       = (cs2_gdf[class_col] == "ice").sum()
count_refrozen  = (cs2_gdf[class_col] == "refrozen").sum()

total = count_lead + count_ice + count_refrozen

density_CS2_lead_only = count_lead / total
density_CS2_leadref   = (count_lead + count_refrozen) / total
density_CS2_floe      = count_ice / total

print("=== CS2 DENSITY ===")
print("Lead only:      ", density_CS2_lead_only)
print("Lead+refrozen:  ", density_CS2_leadref)
print("Floe (ice):     ", density_CS2_floe)
print("Counts:", count_lead, count_ice, count_refrozen, " Total:", total)


=== CS2 DENSITY ===
Lead only:       0.05152514427040396
Lead+refrozen:   0.05152514427040396
Floe (ice):      0.948474855729596
Counts: 125 2301 0  Total: 2426


In [17]:
with rasterio.open(S1_FILE) as src:
    s1_crs = src.crs
    s1_transform = src.transform
    s1_data = src.read(1)

print("S1 loaded. CRS:", s1_crs)
print("S1 shape:", s1_data.shape)


S1 loaded. CRS: EPSG:4326
S1 shape: (13573, 32768)


In [18]:
cs2_proj = cs2_gdf.to_crs(s1_crs)
print("CS2 reprojected.")

CS2 reprojected.


In [19]:
def extract_s1_neighborhoods(s1_array, transform, cs2_points, win=20):
    """
    s1_array : 2D numpy array
    transform : rasterio transform
    cs2_points : GeoSeries of points
    win : window radius (pixels)
    """
    offsets = []
    for point in cs2_points:
        x, y = point.x, point.y
        col, row = ~transform * (x, y)   # map to pixel coordinate
        row, col = int(row), int(col)

        r1, r2 = max(row - win, 0), min(row + win, s1_array.shape[0])
        c1, c2 = max(col - win, 0), min(col + win, s1_array.shape[1])

        patch = s1_array[r1:r2, c1:c2]
        offsets.append(patch.flatten())

    # concatenate all windows
    all_pixels = np.concatenate(offsets)
    return all_pixels


In [20]:
pixels = extract_s1_neighborhoods(
    s1_array=s1_data,
    transform=s1_transform,
    cs2_points=cs2_proj.geometry,
    win=WINDOW_RADIUS
)

valid = pixels[pixels > 0]  # 去掉背景 0

print("Extracted pixels:", len(valid))
print("Unique values:", np.unique(valid))


Extracted pixels: 67376915
Unique values: [1 2 3]


In [21]:
lead_s1 = (valid == S1_LEAD).sum()
ice_s1  = (valid == S1_ICE).sum()
ref_s1  = (valid == S1_REFR).sum()
total_s1 = len(valid)

density_s1_lead_only = lead_s1 / total_s1
density_s1_leadref   = (lead_s1 + ref_s1) / total_s1
density_s1_floe      = ice_s1 / total_s1

print("=== S1 DENSITY (along-track window sampling) ===")
print("Lead only:     ", density_s1_lead_only)
print("Lead+refrozen: ", density_s1_leadref)
print("Floe (ice):    ", density_s1_floe)

print("\nCounts:", lead_s1, ice_s1, ref_s1, "Total:", total_s1)



=== S1 DENSITY (along-track window sampling) ===
Lead only:      0.20072309633054586
Lead+refrozen:  0.6256319838924059
Floe (ice):     0.3743680161075941

Counts: 13524103 25223762 28629050 Total: 67376915


In [23]:
diff_lead_only = density_CS2_lead_only - density_s1_lead_only
diff_leadref   = density_CS2_leadref   - density_s1_leadref
diff_floe      = density_CS2_floe      - density_s1_floe

print("=== DENSITY DIFFERENCES ===")
print("Lead only diff:     ", diff_lead_only)
print("Lead+refrozen diff: ", diff_leadref)
print("Floe diff:          ", diff_floe)


=== DENSITY DIFFERENCES ===
Lead only diff:      -0.14919795206014191
Lead+refrozen diff:  -0.5741068396220019
Floe diff:           0.5741068396220019
